# Joint Cosmos QA / WikiHop fine-tuning on Kaggle

Run cells in order. Edit **only the configuration cell** below. Attach this repository
as a Kaggle Dataset or work from its checkout. Enable a GPU accelerator in Kaggle
settings for full training; CPU fallback supports small verification runs.
Architecture, tokenization, masks,
training, and checkpointing are imported from `bertdm/*.py`.

The two official training splits are sampled jointly; their official validation
splits remain separate. Default sampling assigns each dataset probability 0.5,
independent of its size. Choice order is shuffled only during training. Sequential
RoPE positions make order augmentation useful but do not make the model invariant.

Initial dependency installation, the approximately 982 MB NeoBERT checkpoint, and
missing dataset files require Internet. For offline sessions, attach dependencies
as wheels, the complete model/tokenizer snapshot, and official split files under
`/kaggle/input`; set the corresponding paths and disable downloads. All writes,
including Hugging Face caches, go under `/kaggle/working`.

This notebook contains no claimed benchmark results. Full training starts at the
training cell; use `stop_after_updates=20` for an initial pilot. A tiny backward
smoke test runs first without changing weights.

In [ ]:
CFG = {
    # None auto-detects a checkout or common Kaggle repository attachment paths.
    "repo_dir": None,                       # e.g. "/kaggle/input/bertdm/BertDM"
    "output_dir": "/kaggle/working/bertdm-run",
    "cache_dir": "/kaggle/working/bertdm-preprocessing",
    "read_cache_dirs": [],                  # optional read-only preprocessing caches
    "install_dependencies": True,
    "wheel_dir": None,                      # e.g. "/kaggle/input/bertdm-wheels"
    "allow_downloads": True,
    "device": "auto",                       # auto / cuda / cpu
    "resume_from": None,                    # previous run directory, latest.json, or checkpoint directory
    "stop_after_updates": None,             # absolute optimizer-step limit; None = all configured epochs
    "datasets": {
        "cosmos": {"train": None, "validation": None},
        "wikihop": {"train": None, "validation": None},
    },                                      # None pairs download pinned official-split files
    "model": {
        "checkpoint": "chandar-lab/NeoBERT", # or complete local snapshot under /kaggle/input
        "revision": "5424c8efeea6491b151d62dee55a752165407430",
        "max_length": 512, "head_width": 256, "dropout": 0.1,
        "context_truncation": "head",        # head keeps the beginning; tail keeps the end
    },
    "training": {
        "epochs": 2,
        "micro_batch_size": 1, "eval_batch_size": 1, "gradient_accumulation": 8,
        "encoder_lr": 2e-5, "head_lr": 1e-4, "weight_decay": 0.01,
        "warmup_ratio": 0.06, "max_grad_norm": 1.0,
        "seed": 17, "precision": "auto",    # auto prefers CUDA bf16, else fp16; CPU uses fp32
        "gradient_checkpointing": True, "deterministic": True, "initial_scale": 1024.0,
        "samples_per_epoch": None,          # None = sum of retained training sizes; weighted replacement sampling
        "dataset_weights": {"cosmos": 1.0, "wikihop": 1.0},
        "shuffle_choices": True,
        "choice_count_bounds": (4, 8, 16, 32),
        "selection_dataset": "macro",       # macro / cosmos / wikihop
        "selection_metric": "accuracy",     # accuracy maximized; nll minimized
        "log_every_updates": 10, "save_every_updates": 100,
        "allow_training_rejections": False, "allow_validation_rejections": False,
        "prune_unreferenced_checkpoints": True,
    },
}

## 1. Paths and pinned dependencies

Read-only repository attachments are imported directly; no editable installation
or write into `/kaggle/input` is required. `requirements-kaggle.txt` pins the Python
model/data libraries. Keep Kaggle's CUDA PyTorch wheel (version 2.5 or newer, below
3); its exact version and GPU are recorded. Matching software/hardware is required
for exact resume. Already matching packages need no Internet; otherwise setup uses
PyPI or the configured offline wheel directory. Restart the kernel after changing
versions that were already imported in an earlier interactive run.

In [ ]:
import os
import sys
import json
import subprocess
import importlib.metadata
from pathlib import Path

output_dir = Path(CFG["output_dir"]).resolve()
cache_dir = Path(CFG["cache_dir"]).resolve()
if Path("/kaggle/input").exists():
    working = Path("/kaggle/working").resolve()
    for path in (output_dir, cache_dir):
        if not path.is_relative_to(working):
            raise ValueError("On Kaggle, output_dir and cache_dir must be under /kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(output_dir / "hf-cache")
os.environ["HF_HUB_CACHE"] = str(output_dir / "hf-cache" / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # set before CUDA initialization
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
os.environ["HF_HUB_OFFLINE"] = "0" if CFG["allow_downloads"] else "1"

candidates = [Path(CFG["repo_dir"])] if CFG["repo_dir"] else [Path.cwd(), Path.cwd() / "BertDM", Path.cwd().parent]
if not CFG["repo_dir"] and Path("/kaggle/input").exists():
    candidates += [p.parent.parent for pattern in ("*/bertdm/__init__.py", "*/*/bertdm/__init__.py")
                   for p in Path("/kaggle/input").glob(pattern)]
repo = next((p.resolve() for p in candidates if (p / "bertdm" / "training.py").is_file()), None)
if repo is None:
    raise FileNotFoundError("Attach the current repository and set CFG['repo_dir'] to its root")
requirements = repo / "requirements-kaggle.txt"
missing = []
for line in requirements.read_text().splitlines():
    if not line.strip() or line.startswith("#"):
        continue
    name, version = line.strip().split("==")
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        missing.append(line)
if missing:
    if not CFG["install_dependencies"]:
        raise RuntimeError(f"Missing/mismatched pinned dependencies: {missing}")
    command = [sys.executable, "-m", "pip", "install", "-r", str(requirements)]
    if CFG["wheel_dir"]:
        command += ["--no-index", "--find-links", CFG["wheel_dir"]]
    elif not CFG["allow_downloads"]:
        raise RuntimeError("Offline dependency setup requires attached wheels or matching preinstalled packages")
    subprocess.check_call(command)
sys.path.insert(0, str(repo))
print("Repository:", repo)
print("Writable output:", output_dir)
print("Pinned dependencies:", "already installed" if not missing else "installed")

In [ ]:
import torch
from packaging.version import Version
from bertdm import DecisionConfig, DecisionPipeline
from bertdm.download import download_cosmos, download_wikihop
from bertdm.training_data import prepare_split, check_split_paths
from bertdm.training import (TrainConfig, Trainer, seed_everything, resolve_precision,
                             smoke_test, resolve_checkpoint, evaluate, environment_info, MEMORY_HELP)

if not Version("2.5") <= Version(torch.__version__.split("+")[0]) < Version("3"):
    raise RuntimeError("Use a CUDA-compatible PyTorch >=2.5,<3 runtime")
training_config = TrainConfig(**CFG["training"])
seed_everything(training_config.seed, training_config.deterministic)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") if CFG["device"] == "auto" else torch.device(CFG["device"])
precision = resolve_precision(training_config.precision, device)
print(json.dumps(environment_info(), indent=2))
print("Resolved device / precision:", device, precision)
print("Memory guidance:", MEMORY_HELP)
(output_dir / "notebook-config.json").write_text(json.dumps(CFG, indent=2))
(output_dir / "environment.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True))

## 2. Load the pretrained pipeline or a saved training bundle

The checkpoint must have the inspected NeoBERT parameter layout. Initial loading
creates a fresh shared scoring head; joint fine-tuning trains both encoder and
head. A resume bundle supplies its own weights and tokenizer offline. Resume
settings, preprocessing fingerprints, trainer/model code, precision, and runtime
are checked before restoring optimizer state. Changing the planned epoch count or
sampling/optimization settings starts a new run rather than exact continuation.

In [ ]:
resume_checkpoint = resolve_checkpoint(CFG["resume_from"]) if CFG["resume_from"] else None
if resume_checkpoint:
    pipeline = DecisionPipeline.from_pretrained(resume_checkpoint)
    for key in ("revision", "max_length", "head_width", "dropout", "context_truncation"):
        if getattr(pipeline.model.config, key) != CFG["model"][key]:
            raise ValueError(f"Resume model setting changed: {key}")
else:
    if (output_dir / "latest.json").exists():
        raise ValueError("Output contains a previous run; set resume_from or use a fresh output_dir")
    pipeline = DecisionPipeline.from_checkpoint(
        DecisionConfig(**CFG["model"]), local_files_only=not CFG["allow_downloads"],
        cache_dir=str(output_dir / "hf-cache" / "hub"))
print("Supported checkpoint length:", pipeline.model.backbone.config.max_length)
print("Configured sequence budget:", pipeline.model.config.max_length)
print("Encoder parameters:", sum(p.numel() for p in pipeline.model.backbone.parameters()))
print("Head parameters:", sum(p.numel() for p in pipeline.model.head.parameters()))

## 3. Official splits, preprocessing cache, and rejection audit

Supply both paths for each dataset or leave both `None` to download. Cosmos uses
`train.csv` / `valid.csv`; WikiHop supports original `train.json` / `dev.json` and
native Parquet `train` / `validation`. Validation never enters the training sampler.
Local files remain the caller's responsibility to identify as the official splits.

Preprocessing reserves every complete choice, question, and structural token before
truncating context. No label selects evidence. Rejected rows are recorded with
source indices and reasons; there is no silent choice or record removal. The
training guard below defaults to refusing **any** rejection. Explicitly allowing
validation rejection produces metrics for retained rows, with coverage reported.

Cache identity includes source-file SHA256, tokenizer content/special IDs, pinned
revision, tokenization-library versions, sequence budget/truncation, and preprocessing
source hashes. Choice augmentation is applied after cache loading during training.
Cache manifests and `rejections.jsonl` are auditable; `read_cache_dirs` can reuse
matching caches attached under `/kaggle/input`.

In [ ]:
sources = {}
for name, paths in CFG["datasets"].items():
    if name not in ("cosmos", "wikihop"):
        raise ValueError(f"Unknown dataset: {name}")
    if paths["train"] is None and paths["validation"] is None:
        if not CFG["allow_downloads"]:
            raise ValueError(f"Set local official split paths for offline {name}")
        downloader = download_cosmos if name == "cosmos" else download_wikihop
        sources[name] = downloader(output_dir / "data" / name)
    elif paths["train"] is None or paths["validation"] is None:
        raise ValueError(f"Supply both official split paths for {name}")
    else:
        sources[name] = {split: Path(path) for split, path in paths.items()}
check_split_paths(sources)
datasets = {"train": {}, "validation": {}}
summaries = {"train": {}, "validation": {}}
for name, paths in sources.items():
    for split, path in paths.items():
        rows, report = prepare_split(pipeline, name, split, path, cache_dir,
                                     read_cache_dirs=CFG["read_cache_dirs"])
        datasets[split][name] = rows
        summaries[split][name] = report
        print(name, split, "retained", report["retained"], "rejected", report["rejected"],
              "cache_hit", report["cache_hit"])
(output_dir / "dataset-summaries.json").write_text(json.dumps(summaries, indent=2))

In [ ]:
from IPython.display import display, HTML
from html import escape

def show_table(rows):
    if not rows:
        return
    columns = list(rows[0])
    header = "".join(f"<th>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(c, '')))}</td>" for c in columns) + "</tr>" for row in rows)
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

show_table([dict(dataset=name, split=split, total=r["total"], retained=r["retained"],
                 rejected=r["rejected"], coverage=r["coverage"], **r["token_lengths"])
            for split, reports in summaries.items() for name, r in reports.items()])
show_table([dict(dataset=name, split=split, choices=int(count), examples=frequency)
            for split, reports in summaries.items() for name, r in reports.items()
            for count, frequency in sorted(r["choice_counts"].items(), key=lambda pair: int(pair[0]))])
for split, reports in summaries.items():
    for name, report in reports.items():
        if report["rejected"]:
            print(name, split, json.dumps(report["rejection_reasons"], indent=2))
            print("Ledger:", Path(report["cache_dir"]) / "rejections.jsonl")

## 4. Training policy and tiny forward/backward smoke test

Each epoch draws `samples_per_epoch` examples **with replacement**: choose a dataset
using normalized `dataset_weights`, then a uniform row from that dataset. Planned
and observed per-dataset counts are logged. Equal weights balance expected exposure,
not unique-example coverage. Training permutations move complete choice blocks and
remap labels and group IDs; validation keeps its original order.

The optimizer has separate encoder/head learning rates, with weight decay on
matrix weights and no decay on bias/norm vectors. Linear warmup is followed by
linear decay to zero over the configured full run. Partial final accumulation
windows use example-weighted losses. Nonfinite losses/gradients abort the run.

Non-reentrant gradient checkpointing recomputes the same original blocks with the
same additive mask; math SDPA remains forced. FlashAttention and sequence packing
stay disabled. Model weights and RoPE buffers remain fp32 while CUDA autocast uses
bf16 or fp16. fp16 uses a resumable GradScaler.

For OOM, reduce sequence budget and microbatch size, increase accumulation, and
keep checkpointing enabled. A budget reduction requires new preprocessing and a
new optimization run. Avoid requesting attention/hidden-state diagnostics in full
training. A smaller smoke input does not establish worst-case memory capacity.

In [ ]:
trainer = Trainer(pipeline, datasets, summaries, training_config, output_dir, device)
print("Samples per epoch:", trainer.samples)
print("Dataset weights:", trainer.weights)
print("Effective batch size:", training_config.micro_batch_size * training_config.gradient_accumulation)
print("Planned optimizer updates:", trainer.total_updates)
smoke = smoke_test(pipeline, device, precision, training_config.gradient_checkpointing,
                   training_config.initial_scale)
print("Smoke test:", json.dumps(smoke, indent=2))
if resume_checkpoint:
    # Restore last, after initialization/smoke, so the next dropout RNG draw is exact.
    trainer.resume(resume_checkpoint)

## 5. Joint fine-tuning and separate validation

The next cell performs training. Validation runs at epoch boundaries and at an
explicit pilot stop. Accuracy and mean negative log-likelihood are reported for
Cosmos and WikiHop individually, plus bins `2-4`, `5-8`, `9-16`, `17-32`, `33+`.
NLL is cross-entropy from masked logits; it is not length-normalized or calibrated.
Macro metrics average datasets equally. `selection_dataset` / `selection_metric`
control the best checkpoint; no validation row is used for gradient updates.

Every optimizer update logs loss, per-group learning rates, gradient norm, scale,
dataset exposure, and timing to `history.jsonl`. Latest checkpoints are saved at
optimizer boundaries on the configured cadence and after validation. The best and
latest may refer to the same checkpoint. Only unreferenced completed checkpoints
created by this trainer are pruned when enabled.

In [ ]:
result = trainer.fit(stop_after_updates=CFG["stop_after_updates"])
print(json.dumps(result, indent=2))
print("Latest:", output_dir / "latest.json")
print("Best:", output_dir / "best.json")

In [ ]:
final_metrics = evaluate(pipeline, datasets["validation"], device, precision,
                         training_config.eval_batch_size, training_config.choice_count_bounds)
show_table([dict(dataset=name, **{k: v for k, v in metrics.items() if k != "by_choice_count"},
                 coverage=summaries["validation"][name]["coverage"])
            for name, metrics in final_metrics.items() if name != "macro"])
show_table([dict(dataset=name, choices=bin_name, **metrics)
            for name, report in final_metrics.items() if name != "macro"
            for bin_name, metrics in report["by_choice_count"].items()])
print("Macro:", final_metrics["macro"])
(output_dir / "final-metrics.json").write_text(json.dumps(final_metrics, indent=2))

history = [json.loads(line) for line in (output_dir / "history.jsonl").read_text().splitlines()]
updates = [row for row in history if row["kind"] == "train"]
show_table([dict(step=r["step"], loss=r["loss"], seconds=r["seconds"],
                 examples_per_second=r["examples_per_second"], learning_rates=r["learning_rates"])
            for r in updates[-10:]])
print("Logged optimizer updates in this output directory:", len(updates))

## 6. Persistence and exact continuation between Kaggle sessions

A checkpoint contains the existing inference bundle plus `training_state.pt`:
AdamW moments, scheduler and fp16 scaler, Python/NumPy/Torch/CUDA random states,
epoch, next sampled-example offset, optimizer step, and best-model progress.
Sampling plans and choice permutations regenerate deterministically. Checkpoints
are written only after an entire accumulation window; unsaved work is replayed
from the last saved boundary after interruption. Exact numerical continuation
requires the same code, settings, data/tokenizer fingerprints, precision, and
runtime/GPU. Resume refuses incompatible runs.

`latest.json` and `best.json` contain **relative** paths into `checkpoints/`, so
save the entire output directory. `training_state.pt` uses Python serialization:
load only your own trusted checkpoints. The inference safetensors bundle does not
need that file.

Before a Kaggle session ends, **Save Version with outputs** or download/publish
`/kaggle/working/bertdm-run` as a Kaggle Dataset. Working storage does not persist
into a fresh session automatically. In the next session, attach the saved outputs
under `/kaggle/input`, set `resume_from` to its `latest.json` (or run directory),
set a fresh writable `output_dir`, and restore the original model/training/data
settings. Matching preprocessing caches may also be attached through
`read_cache_dirs`. The previous best is copied into the new output so it remains
available even when later validation does not improve. Keep the previous output's
logs for a full historical record; the new directory logs the resumed segment.

Model+AdamW checkpoints can occupy several GB each. Best/latest are retained;
periodic saving incurs disk I/O and may need a longer cadence on large runs.

## 7. Reload the best bundle and return probabilities plus selected choice

The following inference uses an original-order validation example only for
inspection. It does not update the model. The final in-memory model is released
before reloading to avoid holding two GPU backbones at once.

In [ ]:
import gc
from bertdm.data import read_records, normalize_cosmos, normalize_wikihop, ExampleError

best_path = resolve_checkpoint(output_dir / "best.json")
del trainer, pipeline
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
restored = DecisionPipeline.from_pretrained(best_path, device=str(device))
name = sorted(sources)[0]
normalizer = {"cosmos": normalize_cosmos, "wikihop": normalize_wikihop}[name]
example = None
for raw in read_records(sources[name]["validation"]):
    try:
        candidate = normalizer(raw)
        restored.encode(candidate)
        example = candidate
        break
    except ExampleError:
        continue
if example is None:
    raise ValueError("No retained inference example")
inference = restored.predict([dict(example, label=None)], temperature=1.0)
selected = int(inference["predictions"][0])
answer = {"dataset": name, "question": example["question"],
          "choices": example["choices"],
          "probabilities": inference["probabilities"][0, :len(example["choices"])].cpu().tolist(),
          "selected_index": selected, "selected_choice": example["choices"][selected]}
print(json.dumps(answer, indent=2))
(output_dir / "inference-example.json").write_text(json.dumps(answer, indent=2))

## Validation scope

See `docs/notebook-validation.md` for the checks actually executed when this
notebook was added. Tiny-model end-to-end execution and any full-checkpoint GPU
smoke checks do not constitute full task training or Kaggle benchmark results.
The training artifacts generated by your run provide its actual progress and
validation coverage. Context truncation can remove multi-hop evidence, and
choice-order sensitivity remains part of the sequential-position design.